<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-03/blob/main/hw_3_data__loading_sql_dql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 3 — Data Loading and SQL DQL

## 1. Setup and Data Loading

In [4]:
!pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow ucimlrepo

import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pg = pgserver.get_server('/tmp/hw3_pg', cleanup_mode='stop')
engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)


ERROR: Could not find a version that satisfies the requirement pgserver (from versions: none)
ERROR: No matching distribution found for pgserver


ModuleNotFoundError: No module named 'pgserver'

In [ ]:
from ucimlrepo import fetch_ucirepo

# UCI Seoul Bike Sharing Demand
dataset = fetch_ucirepo(id=560)

df = pd.concat(
    [dataset.data.features, dataset.data.targets],
    axis=1
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

In [ ]:
csv_path = "/content/hw3_seoul_bike.csv"

df.to_csv(csv_path, index=False)

print("CSV saved:", csv_path)
print("Rows:", len(df))

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bike_clean CASCADE;
        DROP TABLE IF EXISTS hw3_bike_staging CASCADE;

        CREATE TABLE hw3_bike_staging (
            date_raw TEXT,
            rented_bike_count_raw TEXT,
            hour_raw TEXT,
            temperature_raw TEXT,
            humidity_raw TEXT,
            wind_speed_raw TEXT,
            visibility_raw TEXT,
            dew_point_temperature_raw TEXT,
            solar_radiation_raw TEXT,
            rainfall_raw TEXT,
            snowfall_raw TEXT,
            seasons_raw TEXT,
            holiday_raw TEXT,
            functioning_day_raw TEXT
        );
    """))

print("Staging table created successfully.")

In [ ]:
raw_conn = engine.raw_connection()

try:
    with raw_conn.cursor() as cur:
        with open(csv_path, "r", encoding="utf-8", newline="") as file:
            cur.copy_expert(
                """
                COPY hw3_bike_staging (
                    date_raw,
                    rented_bike_count_raw,
                    hour_raw,
                    temperature_raw,
                    humidity_raw,
                    wind_speed_raw,
                    visibility_raw,
                    dew_point_temperature_raw,
                    solar_radiation_raw,
                    rainfall_raw,
                    snowfall_raw,
                    seasons_raw,
                    holiday_raw,
                    functioning_day_raw
                )
                FROM STDIN
                WITH (
                    FORMAT csv,
                    HEADER true,
                    DELIMITER ','
                );
                """,
                file
            )

    raw_conn.commit()

finally:
    raw_conn.close()

print("CSV loaded into staging successfully.")

In [ ]:
pd.read_sql("""
    SELECT COUNT(*) AS loaded_rows
    FROM hw3_bike_staging;
""", engine)

The staging table contains 8,760 rows, which matches the source dataset row count. Therefore, all source records were loaded successfully through COPY FROM STDIN.

In [ ]:
pd.read_sql("""
    SELECT
        date_raw,
        rented_bike_count_raw,
        hour_raw,
        temperature_raw,
        seasons_raw,
        holiday_raw,
        functioning_day_raw
    FROM hw3_bike_staging
    LIMIT 5;
""", engine)

## 2. Staging and Typed Table

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        DROP TABLE IF EXISTS hw3_bike_clean CASCADE;

        CREATE TABLE hw3_bike_clean (
            record_id BIGINT
                GENERATED BY DEFAULT AS IDENTITY
                PRIMARY KEY,

            date DATE NOT NULL,

            rented_bike_count INTEGER NOT NULL
                CHECK (rented_bike_count >= 0),

            hour SMALLINT NOT NULL
                CHECK (hour BETWEEN 0 AND 23),

            temperature NUMERIC(3,1),

            humidity SMALLINT
                CHECK (humidity BETWEEN 0 AND 100),

            wind_speed NUMERIC(4,1)
                CHECK (wind_speed >= 0),

            visibility SMALLINT
                CHECK (visibility >= 0),

            dew_point_temperature NUMERIC(4,1),

            solar_radiation NUMERIC(5,2)
                CHECK (solar_radiation >= 0),

            rainfall NUMERIC(5,1)
                CHECK (rainfall >= 0),

            snowfall NUMERIC(4,1)
                CHECK (snowfall >= 0),

            seasons TEXT,

            holiday BOOLEAN,

            functioning_day BOOLEAN NOT NULL
        );
    """))

print("Clean table structure created successfully.")

In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        WITH parsed AS (
            SELECT
                TO_DATE(
                    NULLIF(TRIM(date_raw), ''),
                    'DD/MM/YYYY'
                ) AS date,

                NULLIF(TRIM(rented_bike_count_raw), '')::INTEGER
                    AS rented_bike_count,

                NULLIF(TRIM(hour_raw), '')::SMALLINT
                    AS hour,

                NULLIF(TRIM(temperature_raw), '')::NUMERIC
                    AS temperature,

                NULLIF(TRIM(humidity_raw), '')::SMALLINT
                    AS humidity,

                NULLIF(TRIM(wind_speed_raw), '')::NUMERIC
                    AS wind_speed,

                NULLIF(TRIM(visibility_raw), '')::SMALLINT
                    AS visibility,

                NULLIF(TRIM(dew_point_temperature_raw), '')::NUMERIC
                    AS dew_point_temperature,

                NULLIF(TRIM(solar_radiation_raw), '')::NUMERIC
                    AS solar_radiation,

                NULLIF(TRIM(rainfall_raw), '')::NUMERIC
                    AS rainfall,

                NULLIF(TRIM(snowfall_raw), '')::NUMERIC
                    AS snowfall,

                NULLIF(TRIM(seasons_raw), '')
                    AS seasons,

                CASE TRIM(holiday_raw)
                    WHEN 'Holiday' THEN TRUE
                    WHEN 'No Holiday' THEN FALSE
                    ELSE NULL
                END AS holiday,

                CASE TRIM(functioning_day_raw)
                    WHEN 'Yes' THEN TRUE
                    WHEN 'No' THEN FALSE
                    ELSE NULL
                END AS functioning_day

            FROM hw3_bike_staging
        )

        INSERT INTO hw3_bike_clean (
            date,
            rented_bike_count,
            hour,
            temperature,
            humidity,
            wind_speed,
            visibility,
            dew_point_temperature,
            solar_radiation,
            rainfall,
            snowfall,
            seasons,
            holiday,
            functioning_day
        )

        SELECT
            date,
            rented_bike_count,
            hour,
            temperature,
            humidity,
            wind_speed,
            visibility,
            dew_point_temperature,
            solar_radiation,
            rainfall,
            snowfall,
            seasons,
            holiday,
            functioning_day
        FROM parsed;
    """))

print("Clean data inserted successfully.")

## 3.Data Quality Audit

In [ ]:
pd.read_sql("""

    SELECT 'staging' AS layer, COUNT(*) AS n_rows
    FROM hw3_bike_staging

    UNION ALL

    SELECT 'clean' AS layer, COUNT(*) AS n_rows
    FROM hw3_bike_clean

""", engine)


The number of records remained unchanged after cleaning. All 8,760 staging rows were successfully loaded into the typed clean table.

In [ ]:
pd.read_sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (hour, date)) AS distinct_business_keys,
        COUNT(*) - COUNT(DISTINCT (hour, date)) AS duplicate_rows
    FROM hw3_bike_clean;

""", engine)


In the Seoul Bike dataset, each row represents bike demand for a specific date and hour. Therefore, (date, hour) was used as the business key for the duplicate check. All 8,760 rows have unique date-hour combinations, so no duplicate records were found.

In [ ]:
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(date_raw), '?') IS NULL
        ) AS missing_date,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(hour_raw), '?') IS NULL
        ) AS missing_hour,


        COUNT(*) FILTER (
              WHERE NULLIF (TRIM(holiday_raw), '?') IS NULL
        ) AS missing_holiday,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(functioning_day_raw), '?') IS NULL
        ) AS missing_functioning_day,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(seasons_raw), '?') IS NULL
        ) AS missing_seasons

        FROM hw3_bike_staging;

""", engine)


No ? missing-value markers were found in the selected staging columns. This is consistent with the Seoul Bike dataset being largely complete, but additional checks for empty strings and SQL NULL are still required.

In [ ]:
pd.read_sql("""
      SELECT
          ROUND(COUNT(*) FILTER (
              WHERE temperature IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_temperature,

          ROUND(COUNT(*) FILTER (
              WHERE humidity IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_humidity,

          ROUND(COUNT(*) FILTER (
              WHERE wind_speed IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_wind_speed,

          ROUND(COUNT(*) FILTER (
              WHERE rainfall IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_rainfall,

          ROUND(COUNT(*) FILTER (
              WHERE holiday IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_holiday,

          ROUND(COUNT(*) FILTER (
              WHERE functioning_day IS NULL)* 100.0 / NULLIF(COUNT(*), 0),2
          ) as pct_null_functioning_day,

          COUNT(*) AS total_rows

      FROM hw3_bike_clean;

""", engine)

The selected clean-table columns have a 0% NULL rate across all 8,760 records. No missing values were introduced during the staging-to-clean transformation. However, a zero NULL rate does not by itself guarantee data quality, because categorical columns may still contain unexpected or special values.

In [ ]:
pd.read_sql("""
    SELECT distinct seasons_raw
    FROM hw3_bike_staging

""", engine)


In [ ]:
pd.read_sql("""
    SELECT distinct holiday_raw
    FROM hw3_bike_staging

""", engine)

In [ ]:
pd.read_sql("""
    SELECT distinct functioning_day_raw
    FROM hw3_bike_staging

""", engine)

The categorical staging fields contain only the expected values. seasons_raw contains the four valid seasons, holiday_raw contains only Holiday and No Holiday, and functioning_day_raw contains only Yes and No. No unexpected categorical markers were found, so the Boolean normalization used in the clean table is valid.

In [ ]:
pd.read_sql("""
    SELECT
        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(date_raw), '') IS NULL
        ) AS missing_date,

        COUNT(*) FILTER (
            WHERE NULLIF(TRIM(hour_raw), '') IS NULL
        ) AS missing_hour,


        COUNT(*) FILTER (
              WHERE NULLIF (TRIM(holiday_raw), '') IS NULL
        ) AS missing_holiday,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(functioning_day_raw), '') IS NULL
        ) AS missing_functioning_day,


        COUNT(*) FILTER (
              WHERE NULLIF(TRIM(seasons_raw), '') IS NULL
        ) AS missing_seasons

        FROM hw3_bike_staging;

""", engine)


No empty-string markers were found in the selected staging columns. Together with the previous checks for ?, SQL NULL, and unexpected categorical values, this indicates that the Seoul Bike dataset is complete for the audited fields.

## 4. Distributions and Outliers

In [ ]:
pd.read_sql("""
    SELECT
        seasons,
        COUNT(*) AS n_rows,
        ROUND(
            COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
            2
        ) AS pct_rows
    FROM hw3_bike_clean
    GROUP BY seasons
    ORDER BY n_rows DESC, seasons
    LIMIT 10;
""", engine)

The seasonal distribution is nearly balanced. Spring and Summer each account for 25.21% of the records, while Autumn and Winter account for 24.93% and 24.66%, respectively. No strong class imbalance is present in the seasons column.

In [ ]:
pd.read_sql("""
    SELECT
        COUNT(rented_bike_count) AS n_non_null,
        MIN(rented_bike_count) AS min_value,
        MAX(rented_bike_count) AS max_value,
        ROUND(AVG(rented_bike_count)::NUMERIC, 2) AS avg_value,
        ROUND(STDDEV_SAMP(rented_bike_count)::NUMERIC, 2) AS sd_value
    FROM hw3_bike_clean;
""", engine)

The average hourly bike demand is about 704.6 rentals, with a standard deviation of 645.0. The large standard deviation indicates substantial variability in demand across hours. The observed values range from 0 to 3,556 rentals.

In [ ]:
pd.read_sql("""

WITH stats AS (
    SELECT
        AVG(rented_bike_count) AS mean_value,
        STDDEV_SAMP(rented_bike_count) AS sd_value
    FROM hw3_bike_clean
    WHERE rented_bike_count IS NOT NULL
)
SELECT
    t.*,
    ROUND(
        ((t.rented_bike_count - s.mean_value) / NULLIF(s.sd_value, 0))::NUMERIC,
        2
    ) AS z_score
FROM hw3_bike_clean AS t
CROSS JOIN stats AS s
WHERE t.rented_bike_count IS NOT NULL
  AND s.sd_value > 0
  AND ABS(t.rented_bike_count - s.mean_value) > 3 * s.sd_value
ORDER BY ABS(t.rented_bike_count - s.mean_value) DESC
LIMIT 20;
""", engine)

The 3σ check identified several unusually high hourly rental counts, with z-scores above 3. Many of the largest values occur around 18:00, suggesting that they may represent real evening demand peaks rather than data errors. Therefore, these observations should be treated as potential outliers, but not removed automatically.

## 5. DQL_EDA Queries

### 5.1 Which hours have the highest number of bike rentals?

In [ ]:
pd.read_sql("""
    SELECT
        hour,
        SUM(rented_bike_count) AS total_rentals
    FROM hw3_bike_clean
    GROUP BY hour
    ORDER BY total_rentals DESC
    LIMIT 10

""", engine)

The highest rental demand occurs at 18:00, with 548,568 rentals, followed by 19:00 and 17:00. A second noticeable peak occurs at 08:00. This pattern suggests that bike usage is strongly associated with morning and evening commuting hours.

### 5.2 How does bike demand change across seasons?

In [ ]:
pd.read_sql("""
    SELECT
        seasons,
        SUM(rented_bike_count) as sum_rent,
        ROUND(AVG(rented_bike_count),2) as avg_rent
    FROM hw3_bike_clean
        GROUP BY seasons
        ORDER BY sum_rent DESC

""", engine)

Bike demand shows a strong seasonal pattern. Summer has the highest total and average rental demand, with an average of 1,034.07 rentals per hour, while Winter has the lowest average at 225.54. This suggests that warmer seasons are associated with substantially higher bike usage.

### 5.3 How does demand differ between holidays and non-holidays?

In [ ]:
pd.read_sql("""
    SELECT
      holiday,
      COUNT(*),
      SUM(rented_bike_count) as sum_rent,
      ROUND(AVG(rented_bike_count),2) as avg_rent
    FROM hw3_bike_clean
    GROUP BY holiday

""", engine)

Non-holiday periods have higher bike demand than holidays. The average hourly rental count is 715.23 on non-holidays compared with 499.76 on holidays. The total number of rentals is also much higher for non-holidays, although this is partly explained by the much larger number of non-holiday observations.

### 5.4 Are there zero-rental records on functioning days, and at what hours do they occur?

In [ ]:
pd.read_sql("""


""", engine)

In [ ]:
pd.read_sql("""


""", engine)

In [ ]:
pd.read_sql("""


""", engine)

In [ ]:
pd.read_sql("""


""", engine)

In [ ]:
pd.read_sql("""


""", engine)

In [ ]:
pd.read_sql("""


""", engine)